# NB09 — Bulk-cohort scoring and admixture covariates
**Run after NB10** (it needs the frozen S, M, R).

For each survival cohort (TCGA = training; CGGA693 and CGGA325 = validation) this notebook computes the frozen scores
S, M and R plus the admixture covariates:
* `immune_stromal`, `neuron_identity` (S genes removed from the marker list), `oligodendrocyte`, `astrocyte_normal`
  marker scores;
* purity proxies `purity_proxy_1 = -immune_stromal` and `purity_proxy_2 = -(oligodendrocyte + astrocyte_normal) / 2`
  (marker-based **proxies**, not DNA-based purity).

Scores are mean z-scores **within each cohort**, using only that cohort's analysed patients. Nothing is refitted.
Also reported: how strongly S, M and R track the admixture covariates (the quantity H1 is about).
Possible CGGA693/CGGA325 duplicate patients (NB02/NB03 screen) are removed from CGGA325.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from gbm_extra import load_expr
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB09 needs frozen gene sets from NB10 and the full cohorts.")
frozen = verify_frozen(); print("Using definitions frozen at", frozen["frozen_at_utc"])
MAT = P("meta", "matrices"); VAL = P("validation")
S = pd.read_csv(P("final", "S_candidate_genes.csv")).gene.astype(str).tolist()
M = pd.read_csv(P("final", "M_genes.csv")).gene.astype(str).tolist()
R = pd.read_csv(P("final", "R_genes.csv")).gene.astype(str).tolist()
mk = CFG["markers"]; Sset = set(S)
neuron_markers = [g for g in mk["neuron_identity"] if g not in Sset]
print("Neuron-identity markers kept (S genes removed):", neuron_markers)
require(len(neuron_markers) >= 3, "Fewer than 3 neuron-identity markers remain after removing S genes; extend config.markers.neuron_identity.")

In [ ]:
# possible duplicate patients between CGGA batches (drop from CGGA325, the second batch)
dup_drop = set()
dup_f = P("meta", "CGGA_possible_duplicate_pairs.csv", mkdir=False)
if dup_f.exists() and dup_f.stat().st_size > 0:
    try:
        dp = pd.read_csv(dup_f)
        print("Possible CGGA duplicate pairs (from NB03):\n", dp.head(20).to_string(index=False))
        ids325 = set(pd.read_csv(P("meta", "CGGA325_samples.csv")).sample_id.astype(str))
        dup_drop = {v for c in dp.columns for v in dp[c].astype(str) if v in ids325}
    except pd.errors.EmptyDataError:
        pass
print("CGGA325 samples dropped as possible duplicates of CGGA693:", sorted(dup_drop))

cohort_files = {"TCGA": MAT / "TCGA_logcpm.csv.gz", "CGGA693": MAT / "CGGA693_log2fpkm.csv.gz", "CGGA325": MAT / "CGGA325_log2fpkm.csv.gz"}
score_tables, cov_report = {}, []
for name, f in cohort_files.items():
    if not f.exists():
        log(f"{name}: matrix not found; cohort skipped.", "WARN")
        continue
    sheet = pd.read_csv(P("meta", f"{name}_samples.csv"))
    flag = "included_survival" if name == "TCGA" else "included"
    require(flag in sheet.columns, f"{name}: column '{flag}' missing; run NB04 part A2 (TCGA) / NB02 (CGGA).")
    ids = sheet.loc[sheet[flag].astype(bool), "sample_id"].astype(str)
    if name == "CGGA325":
        ids = ids[~ids.isin(dup_drop)]
    e = load_expr(f)
    ids = [s for s in ids if s in e.columns]
    require(len(ids) >= 30, f"{name}: only {len(ids)} analysed samples found in the matrix.")
    e = e[ids]; e = e[e.std(axis=1) > 0]
    z = zscore_df(e, axis=1)
    df = pd.DataFrame({"sample_id": ids})
    for lab, genes in [("S", S), ("M", M), ("R", R)]:
        df[lab] = score_geneset(z, genes).values
    df["immune_stromal"] = score_geneset(z, mk["immune_stromal"]).values
    df["neuron_identity"] = score_geneset(z, neuron_markers).values
    df["oligodendrocyte"] = score_geneset(z, mk["oligodendrocyte"]).values
    df["astrocyte_normal"] = score_geneset(z, mk["astrocyte_normal"]).values
    df["purity_proxy_1"] = -df.immune_stromal
    df["purity_proxy_2"] = -(df.oligodendrocyte + df.astrocyte_normal) / 2
    keep_cols = [c for c in ["age", "sex", "os_time_days", "os_event", "mgmt", "radio", "chemo", "cohort", "idh_status"] if c in sheet.columns]
    clin = sheet.drop_duplicates("sample_id").set_index("sample_id").loc[ids, keep_cols].reset_index()
    df = df.merge(clin, on="sample_id", how="left")
    n_found = {k: len([g for g in v if g in e.index]) for k, v in {"S": S, "M": M, "R": R, "neuron": neuron_markers}.items()}
    print(f"{name}: n={len(df)}, events={int(df.os_event.sum())}, genes found = {n_found}")
    require(min(n_found["S"], n_found["M"], n_found["R"]) >= 10, f"{name}: fewer than 10 genes of a frozen set are measured.")
    require(df[["S", "M", "R", "neuron_identity", "purity_proxy_1", "purity_proxy_2"]].notna().all().all(),
            f"{name}: a score is missing (fewer than 3 marker genes measured).")
    df.to_csv(VAL / f"{name}_scores.csv", index=False)
    score_tables[name] = df
    for sc in ["S", "M", "R"]:
        for cv in ["immune_stromal", "neuron_identity", "oligodendrocyte", "astrocyte_normal", "purity_proxy_1", "purity_proxy_2"]:
            rho, p = stats.spearmanr(df[sc], df[cv], nan_policy="omit")
            cov_report.append({"cohort": name, "score": sc, "covariate": cv, "n": len(df), "spearman_rho": rho, "p": p})
    rho_mr = stats.spearmanr(df.M, df.R)[0]
    print(f"   Spearman rho(M, R) = {rho_mr:.2f}")
    manifest_update("validation", name, {"n": int(len(df)), "events": int(df.os_event.sum()), "genes_found": n_found, "rho_M_R": float(rho_mr)})
require(len(score_tables) >= 2, f"Need at least 2 scored cohorts; got {list(score_tables)}.")
cov = pd.DataFrame(cov_report); cov.to_csv(VAL / "score_vs_admixture_correlations.csv", index=False)
print(cov.pivot_table(index=["cohort", "score"], columns="covariate", values="spearman_rho").round(2).to_string())

In [ ]:
names = list(score_tables)
fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 3.8), squeeze=False)
for ax, n in zip(axes[0], names):
    d = score_tables[n]
    for sc, c in [("M", "#1f3a93"), ("R", "#0f9d94")]:
        rho = stats.spearmanr(d[sc], d.neuron_identity, nan_policy="omit")[0]
        ax.scatter(d.neuron_identity, d[sc], s=7, alpha=.5, c=c, label=f"{sc} (rho={rho:.2f})")
    ax.set_xlabel("normal neuron-identity marker score"); ax.set_ylabel("gene-set score"); ax.set_title(n); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "Scores_vs_neuron_identity", folder="validation"); plt.close(fig)
print("\nNB09 complete. Next: NB11 (survival and H1 verdict).")